In [1]:
# Hands-On Lab: Sentiment Analysis with PyTorch

import re
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.nn.utils.rnn import pad_sequence

# --------------------------------------------------
# 1. Create a small text dataset
# --------------------------------------------------

data = {
    "text": [
        "I love this product",
        "This movie was fantastic",
        "The service was excellent",
        "I hate this product",
        "The experience was terrible",
        "This movie was awful"
    ],
    "label": [
        1, 1, 1,
        0, 0, 0
    ]
}

df = pd.DataFrame(data)
print(df)


                          text  label
0          I love this product      1
1     This movie was fantastic      1
2    The service was excellent      1
3          I hate this product      0
4  The experience was terrible      0
5         This movie was awful      0


In [2]:

# --------------------------------------------------
# 2. Clean and tokenize text
# --------------------------------------------------

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    return tokens

df["tokens"] = df["text"].apply(preprocess)
print(df[["text", "tokens"]])


                          text                            tokens
0          I love this product          [i, love, this, product]
1     This movie was fantastic     [this, movie, was, fantastic]
2    The service was excellent    [the, service, was, excellent]
3          I hate this product          [i, hate, this, product]
4  The experience was terrible  [the, experience, was, terrible]
5         This movie was awful         [this, movie, was, awful]


In [3]:

# --------------------------------------------------
# 3. Create vocabulary dictionary
# --------------------------------------------------

vocab = {"<PAD>": 0, "<UNK>": 1}

for tokens in df["tokens"]:
    for word in tokens:
        if word not in vocab:
            vocab[word] = len(vocab)

print(vocab)


{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'this': 4, 'product': 5, 'movie': 6, 'was': 7, 'fantastic': 8, 'the': 9, 'service': 10, 'excellent': 11, 'hate': 12, 'experience': 13, 'terrible': 14, 'awful': 15}


In [4]:

# --------------------------------------------------
# 4. Convert text to numerical sequences
# --------------------------------------------------

def text_to_sequence(tokens):
    return [vocab.get(word, vocab["<UNK>"]) for word in tokens]

df["sequence"] = df["tokens"].apply(text_to_sequence)
print(df[["tokens", "sequence"]])


                             tokens        sequence
0          [i, love, this, product]    [2, 3, 4, 5]
1     [this, movie, was, fantastic]    [4, 6, 7, 8]
2    [the, service, was, excellent]  [9, 10, 7, 11]
3          [i, hate, this, product]   [2, 12, 4, 5]
4  [the, experience, was, terrible]  [9, 13, 7, 14]
5         [this, movie, was, awful]   [4, 6, 7, 15]


In [5]:

# --------------------------------------------------
# 5. Pad sequences
# --------------------------------------------------

sequences = [
    torch.tensor(seq, dtype=torch.long)
    for seq in df["sequence"]
]

X = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=vocab["<PAD>"]
)

y = torch.tensor(df["label"].values, dtype=torch.long)

print("Padded Input Tensor:")
print(X)

print("Labels:")
print(y)


Padded Input Tensor:
tensor([[ 2,  3,  4,  5],
        [ 4,  6,  7,  8],
        [ 9, 10,  7, 11],
        [ 2, 12,  4,  5],
        [ 9, 13,  7, 14],
        [ 4,  6,  7, 15]])
Labels:
tensor([1, 1, 1, 0, 0, 0])


In [6]:

# --------------------------------------------------
# 6. Build the sentiment classification model
# --------------------------------------------------

class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.fc = nn.Linear(
            embedding_dim,
            num_classes
        )

    def forward(self, x):
        embedded = self.embedding(x)
        pooled = embedded.mean(dim=1)
        output = self.fc(pooled)
        return output

vocab_size = len(vocab)
embedding_dim = 50
num_classes = 2

model = SentimentClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_classes=num_classes
)

print(model)


SentimentClassifier(
  (embedding): Embedding(16, 50, padding_idx=0)
  (fc): Linear(in_features=50, out_features=2, bias=True)
)


In [ ]:

# --------------------------------------------------
# 7. Train the model
# --------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{num_epochs}], "
            f"Loss: {loss.item():.4f}"
        )


In [7]:

# --------------------------------------------------
# 8. Evaluate the model
# --------------------------------------------------

model.eval()

with torch.no_grad():
    outputs = model(X)

    predictions = torch.argmax(outputs, dim=1)

    accuracy = (predictions == y).float().mean()

print("Predictions:", predictions.tolist())
print("Actual Labels:", y.tolist())
print(f"Training Accuracy: {accuracy.item() * 100:.2f}%")


Predictions: [0, 0, 1, 1, 0, 0]
Actual Labels: [1, 1, 1, 0, 0, 0]
Training Accuracy: 50.00%


In [10]:

# --------------------------------------------------
# 9. Predict sentiment for new examples
# --------------------------------------------------

def predict_sentiment(text):
    model.eval()

    tokens = preprocess(text)

    sequence = [
        vocab.get(word, vocab["<UNK>"])
        for word in tokens
    ]

    input_tensor = torch.tensor(
        sequence,
        dtype=torch.long
    ).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        prediction = torch.argmax(output, dim=1).item()

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"


In [9]:

# --------------------------------------------------
# 10. Test new sentences
# --------------------------------------------------

examples = [
    "This product is amazing",
    "I am very disappointed",
    "The movie was excellent",
    "This service was awful"
]

for example in examples:
    sentiment = predict_sentiment(example)
    print(f"Text: {example}")
    print(f"Predicted Sentiment: {sentiment}")
    print()

Text: This product is amazing
Predicted Sentiment: Negative

Text: I am very disappointed
Predicted Sentiment: Positive

Text: The movie was excellent
Predicted Sentiment: Negative

Text: This service was awful
Predicted Sentiment: Negative

